# Verify setup (read-only)
- Checks this company's three environments and the shared account items against the design: storage, volumes, schemas, grants (access matrix `src/lh/access.py`, the same one `01_grant_access` applies), groups and service principals, lock-down, the token vault, secret scopes, and your own tokens and Git credential.
- Run as Management (workspace admin): Run all. Changes nothing, writes nothing. Prints no emails, IDs, keys or values. The last cell lists every WARN and FAIL.
- Not covered here (own checks): jobs = `checks/t28_jobs`, each person's access = `checks/t29_role_access`, masks = `checks/t15_masks_applied`.

In [0]:
import os, sys, datetime
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "src"))
import yaml
from lh import access, common
from databricks.sdk import WorkspaceClient

ROOT = common.repo_root()
company = yaml.safe_load(open(f"{ROOT}/config/company.yml"))
CODE, COMPANY_NAME = company["code"], company["name"]
jobs_cfg = yaml.safe_load(open(f"{ROOT}/config/jobs.yml"))
ENVS = ["dev", "test", "prod"]
SCHEMAS = {"bronze", "silver", "gold", "ops"}
VOLUMES = {                       # volume -> (schema, type, environments)
    "landing":     ("bronze", "EXTERNAL", {"dev", "test", "prod"}),
    "processed":   ("bronze", "EXTERNAL", {"dev", "test", "prod"}),
    "rejected":    ("bronze", "EXTERNAL", {"dev", "test", "prod"}),
    "checkpoints": ("ops",    "MANAGED",  {"dev", "test", "prod"}),
    "refresh_in":  ("ops",    "MANAGED",  {"dev", "test"}),
    "refresh_out": ("ops",    "MANAGED",  {"test", "prod"}),
}
w = WorkspaceClient()
results = []
def check(area, env, name, ok, detail="", level="FAIL"):
    results.append((area, env, name, "PASS" if ok else level, str(detail)[:180]))
sps = {s.display_name: s for s in w.service_principals.list()}
app_name = {s.application_id: n for n, s in sps.items()}
id_name = {s.id: n for n, s in sps.items()}
all_catalogs = {c.name for c in w.catalogs.list()}
print(f"{COMPANY_NAME}: checking {', '.join(f'{CODE}_{e}' for e in ENVS)} + shared account items (read-only)")

## 1. Storage credentials, external locations, volumes (read)

In [0]:
for env in ENVS:
    cred, loc, bucket = f"lh_{CODE}_{env}_cred", f"lh_{CODE}_{env}_source", f"lh-{CODE}-{env}-source"
    try:
        c = w.storage_credentials.get(cred)
        arn = c.aws_iam_role.role_arn if c.aws_iam_role else ""
        check("storage", env, f"credential {cred} uses its own IAM role", arn.endswith(f":role/lh-{CODE}-{env}-databricks"),
              "role " + arn.split(":role/")[-1])
        l = w.external_locations.get(loc)
        check("storage", env, f"location {loc} = own bucket with own credential",
              (l.url or "").rstrip("/") == f"s3://{bucket}" and l.credential_name == cred, l.url)
    except Exception as e:
        check("storage", env, f"credential and location visible", False, f"{type(e).__name__}: needs metastore admin", level="WARN")
    cat = f"{CODE}_{env}"
    if cat not in all_catalogs:
        check("catalog", env, f"catalog {cat} exists", False)
        continue
    check("catalog", env, f"catalog {cat} exists", True)
    schemas = {s.name for s in w.schemas.list(catalog_name=cat)} - {"information_schema"}
    check("catalog", env, "schemas exactly bronze/silver/gold/ops", schemas == SCHEMAS,
          f"missing {sorted(SCHEMAS - schemas) or 'none'}, extra {sorted(schemas - SCHEMAS) or 'none'}")
    rows = spark.sql(f"""SELECT volume_schema, volume_name, volume_type, storage_location
                         FROM system.information_schema.volumes WHERE volume_catalog = '{cat}'""").collect()
    actual = {(r.volume_schema, r.volume_name): r for r in rows}
    want = {(s, v) for v, (s, t, envs) in VOLUMES.items() if env in envs}
    check("volume", env, "volumes exactly as designed", set(actual) == want,
          f"missing {sorted(want - set(actual)) or 'none'}, extra {sorted(set(actual) - want) or 'none'}")
    for (s, v), r in sorted(actual.items()):
        if v not in VOLUMES:
            continue
        vtype = VOLUMES[v][1]
        ok = r.volume_type == vtype and (vtype == "MANAGED" or (r.storage_location or "").rstrip("/") == f"s3://{bucket}/{v}")
        check("volume", env, f"{s}.{v} is {vtype}" + (f" on s3://{bucket}/{v}" if vtype == "EXTERNAL" else ""), ok, r.volume_type)
        try:
            dbutils.fs.ls(f"/Volumes/{cat}/{s}/{v}/")
            check("volume", env, f"{s}.{v} readable (live)", True)
        except Exception as e:
            check("volume", env, f"{s}.{v} readable (live)", False, type(e).__name__)
print("Storage, catalogs, schemas and volumes checked")

## 2. Grants = access matrix (`src/lh/access.py`) + controlled-refresh grants (`03_refresh_setup`)

In [0]:
norm = lambda a: a.replace("_", " ").upper()
def direct_grants(kind, obj):
    rows = spark.sql(f"SHOW GRANTS ON {kind} {obj}").collect()
    who = lambda p: "a person" if "@" in p else app_name.get(p, p)          # never print an address
    return {(who(r.Principal), norm(r.ActionType)) for r in rows if r.ObjectType.upper() == kind}

for env in ENVS:
    cat = f"{CODE}_{env}"
    if cat not in all_catalogs:
        continue
    want = {k: {(n, norm(p)) for n, p in v} for k, v in access.expected(CODE, env).items()}
    for s in SCHEMAS:
        want.setdefault(("SCHEMA", f"{cat}.{s}"), set())          # schemas without a row in the matrix: no direct grants
    for (kind, obj), exp in sorted(want.items()):
        act = direct_grants(kind, obj)
        check("grants", env, f"{kind.lower()} {obj.split('.', 1)[-1] if kind != 'CATALOG' else obj} grants as designed",
              act == exp, f"missing {sorted(exp - act) or 'none'}, extra {sorted(act - exp) or 'none'}")
    vols = spark.sql(f"SELECT volume_schema, volume_name FROM system.information_schema.volumes WHERE volume_catalog = '{cat}'").collect()
    stray = [f"{r.volume_schema}.{r.volume_name}" for r in vols
             if ("VOLUME", f"{cat}.{r.volume_schema}.{r.volume_name}") not in want
             and direct_grants("VOLUME", f"{cat}.{r.volume_schema}.{r.volume_name}")]
    check("grants", env, "no other direct grants on volumes", not stray, stray or "none")
    writers = {p for p, a in direct_grants("CATALOG", cat) if a in ("MODIFY", "CREATE TABLE", "WRITE VOLUME")}
    exp_writers = {n for n, sc, pr in access.matrix(CODE, env) if sc == "catalog" and "MODIFY" in pr}
    check("grants", env, "only designed principals can write", writers == exp_writers, sorted(writers))
print("Grants checked")

## 3. Groups, admins, service principals (counts and names of groups/accounts only)

In [0]:
def group(name):
    g = list(w.groups.list(filter=f"displayName eq '{name}'", attributes="id,displayName,members"))
    return g[0] if g else None
members = {}
for name in ["admins", "lh_developer", "lh_teamlead", "lh_manager", "lh_bizmanager", "lh_management",
             "lh_ci", "lh_deploy", "lh_group_reader"]:
    g = group(name)
    if name != "admins":
        check("identity", "all", f"group {name} exists", g is not None)
    members[name] = {m.value for m in (g.members or [])} if g else set()
for name in ["lh_developer", "lh_teamlead", "lh_manager", "lh_bizmanager", "lh_management"]:
    check("identity", "all", f"{name} has members", len(members[name]) >= 1, f"{len(members[name])} member(s)")
AUTOMATION = {"lh_ci": {"sp_lh_notices"}, "lh_deploy": {"sp_lh_deploy"}, "lh_group_reader": {"sp_bhg_test", "sp_bhg_prod"}}
for name, want in AUTOMATION.items():
    got = {id_name.get(m, "a person") for m in members[name]}
    check("identity", "all", f"{name} members = {', '.join(sorted(want))}", got == want, sorted(got))
admins = members["admins"]
check("identity", "all", "Management is workspace admin", bool(members["lh_management"]) and members["lh_management"] <= admins)
check("identity", "all", "admins = Management + break-glass owner only (2 or fewer)", len(admins) <= 2,
      f"{len(admins)} admin(s)", level="WARN")
for name in ["lh_developer", "lh_teamlead", "lh_manager", "lh_bizmanager"]:
    check("identity", "all", f"{name} members are not admins", not (members[name] & admins))
for name, ents in [(f"sp_{CODE}_test", {"workspace-access", "databricks-sql-access"}),
                   (f"sp_{CODE}_prod", {"workspace-access", "databricks-sql-access"}),
                   ("sp_lh_deploy", {"workspace-access"}), ("sp_lh_notices", {"workspace-access", "databricks-sql-access"})]:
    if name not in sps:
        check("identity", "all", f"service principal {name} exists", False)
        continue
    sp = w.service_principals.get(sps[name].id)
    got = {e.value for e in (sp.entitlements or [])}
    check("identity", "all", f"{name} active, not admin", bool(sp.active) and sp.id not in admins)
    check("identity", "all", f"{name} entitlements within {sorted(ents)}", got <= ents, sorted(got))
print("Groups, admins and service principals checked")

## 4. Lock-down (other catalogs hidden; project catalogs not open to all users)

In [0]:
bad = lambda p: p == "account users" or p.startswith("_workspace_users")
for cat in sorted(all_catalogs):
    if cat in ("system", "samples") or (cat.startswith(("bhg_", "fqf_")) and not cat.startswith(f"{CODE}_")):
        continue
    try:
        open_to = sorted({r.Principal for r in spark.sql(f"SHOW GRANTS ON CATALOG `{cat}`").collect() if bad(r.Principal)})
        check("lockdown", "all", f"catalog {cat} not open to all users", not open_to, open_to or "none")
    except Exception as e:
        check("lockdown", "all", f"catalog {cat} not open to all users", False, type(e).__name__, level="WARN")
print("Lock-down checked")

## 4b. Vault (keyed token masks, ADR 0010; set up by `04_vault_setup`)

In [0]:
# Only each job account, only on its own schema lh_vault.<code>_<env>; no person or role group anywhere in the vault.
# The token key is counted, never read or printed.
if access.VAULT not in all_catalogs:
    check("vault", "all", f"catalog {access.VAULT} exists", False, "run setup_once/notebook/04_vault_setup")
else:
    want = {}
    for p, (kind, obj), privs in access.vault_grants():
        want.setdefault((kind, obj), set()).update((p, norm(x)) for x in privs)
    for (kind, obj), exp in sorted(want.items()):
        try:
            act = direct_grants(kind, obj)
        except Exception as e:
            check("vault", "all", f"{kind.lower()} {obj} grants as designed", False, type(e).__name__)
            continue
        check("vault", "all", f"{kind.lower()} {obj} grants as designed", act == exp,
              f"missing {sorted(exp - act) or 'none'}, extra {sorted(act - exp) or 'none'}")
    schemas = {s.name for s in w.schemas.list(catalog_name=access.VAULT)} - {"information_schema", "default"}
    planned = {o.split(".")[1] for k, o in want if k == "SCHEMA"}
    check("vault", "all", "vault schemas = one per job account", schemas == planned,
          f"missing {sorted(planned - schemas) or 'none'}, extra {sorted(schemas - planned) or 'none'}")
    for env in ("test", "prod"):
        v = f"{access.VAULT}.{CODE}_{env}"
        try:   # names only: the key table and the token function are made by the job account on its first run
            tables = {r.tableName for r in spark.sql(f"SHOW TABLES IN {v}").collect()}
            funcs = {r[0].split(".")[-1] for r in spark.sql(f"SHOW USER FUNCTIONS IN {v}").collect()}
            check("vault", env, f"{CODE}_{env} token key and token function exist",
                  "mask_key" in tables and "mask_token" in funcs, f"tables {sorted(tables)}, functions {sorted(funcs)}")
        except Exception as e:
            check("vault", env, f"{CODE}_{env} token key and token function exist", False, type(e).__name__, level="WARN")
        try:   # even Management may not read the key: it belongs to the job account (catalog ownership is not enough)
            spark.sql(f"SELECT count(*) FROM {v}.mask_key").first()
            check("vault", env, f"{CODE}_{env} token key not readable by you", False, "it answered: remove the access")
        except Exception:
            check("vault", env, f"{CODE}_{env} token key not readable by you", True)
print("Vault checked")

## 5. Secret scopes: keys present and access as designed (names only, never a value)

In [0]:
mail_roles = {"dev": set(jobs_cfg["dev_roles"].values()),
              "test": set(jobs_cfg["notify"]["test"]) | {"management"},
              "prod": set(jobs_cfg["notify"]["prod"]) | {"management"}}
KAFKA = {"kafka_bootstrap", "kafka_ca_cert", "kafka_password"}
SCOPES = {   # scope -> (keys that must be there, READ principals, WRITE principals); written by setup-secret-scopes
    "lh_dev":  ({"group_match_key"} | KAFKA | {f"mail_{r}" for r in mail_roles["dev"]},
                {"lh_developer", "lh_teamlead", "lh_manager"}, set()),
    "lh_test": ({"group_match_key"} | KAFKA | {f"mail_{r}" for r in mail_roles["test"]},
                {"sp_bhg_test", "sp_fqf_test", "lh_deploy"}, set()),
    "lh_prod": ({"group_match_key"} | KAFKA | {f"mail_{r}" for r in mail_roles["prod"]},
                {"sp_bhg_prod", "sp_fqf_prod", "lh_deploy"}, set()),
    "lh_refresh": ({"refresh_key"}, {"sp_bhg_prod", "sp_bhg_test", "sp_fqf_prod", "sp_fqf_test"}, {"lh_deploy"}),
}
existing = {s.name for s in w.secrets.list_scopes()}
lengths, same = {}, None
for scope, (keys, readers, writers) in SCOPES.items():
    env = scope.split("_")[1] if scope != "lh_refresh" else "all"
    check("secrets", env, f"scope {scope} exists", scope in existing)
    if scope not in existing:
        continue
    have = {k.key for k in w.secrets.list_secrets(scope=scope)}
    check("secrets", env, f"{scope} has its keys", keys <= have, f"missing {sorted(keys - have) or 'none'}")
    extra_mail = sorted(k for k in have if k.startswith("mail_") and k not in keys)
    check("secrets", env, f"{scope} holds no mail key it does not need", not extra_mail, extra_mail or "none")
    acl = {}
    for a in w.secrets.list_acls(scope=scope):
        who = "a person (admin)" if "@" in a.principal else app_name.get(a.principal, a.principal)
        acl.setdefault(a.permission.value, set()).add(who)
    check("secrets", env, f"{scope} READ as designed", acl.get("READ", set()) == readers, sorted(acl.get("READ", set())))
    check("secrets", env, f"{scope} WRITE as designed", acl.get("WRITE", set()) == writers, sorted(acl.get("WRITE", set())))
    check("secrets", env, f"{scope} MANAGE only by the admin person", acl.get("MANAGE", set()) <= {"a person (admin)"},
          sorted(acl.get("MANAGE", set())))
# the three match keys must differ (the prod key never reused in dev or test); compared here, never printed
vals = {}
for env in ENVS:
    try:
        vals[env] = dbutils.secrets.get(f"lh_{env}", "group_match_key")
        check("secrets", env, f"lh_{env} group_match_key 32+ characters", len(vals[env]) >= 32, f"{len(vals[env])} chars")
    except Exception as e:
        check("secrets", env, f"lh_{env} group_match_key readable by admin", False, type(e).__name__, level="WARN")
if len(vals) == 3:
    check("secrets", "all", "dev/test/prod match keys all different", len(set(vals.values())) == 3)
vals = None   # drop the values from memory
print("Secret scopes checked")

## 6. Your own tokens and Git credential (the person running this)

In [0]:
now_ms = datetime.datetime.now().timestamp() * 1000
toks = list(w.tokens.list())
old = [t.comment for t in toks if (t.comment or "").startswith("lh_github_")]
check("tokens", "all", "no old GitHub-Actions token (revoked 18 Oct, CI uses service principals)", not old, old or "none")
for t in toks:
    days = int(((t.expiry_time or 0) - now_ms) / 86400000) if (t.expiry_time or 0) > 0 else None
    check("tokens", "all", f"token '{t.comment or 'no comment'}' expires and has 14+ days left",
          days is not None and days >= 14, "never expires" if days is None else f"{days} days left", level="WARN")
creds = [str(getattr(c, "git_provider", "")) for c in w.git_credentials.list()]
check("git", "all", "exactly one GitHub credential for this login", sum("github" in p.lower() for p in creds) == 1, creds)
print("Tokens and Git credential checked")

In [0]:
df = spark.createDataFrame(results, "area string, env string, check string, status string, detail string")
display(df.orderBy("status", "area", "env"))
counts = {s: sum(1 for r in results if r[3] == s) for s in ("PASS", "WARN", "FAIL")}
print(f"{COMPANY_NAME}: {len(results)} checks | PASS {counts['PASS']} | WARN {counts['WARN']} | FAIL {counts['FAIL']}")
for r in results:
    if r[3] != "PASS":
        print(f"  {r[3]:4} [{r[0]}/{r[1]}] {r[2]} -> {r[4]}")